# E1 · W6 · Kaggle run 16 — xuất 40 ảnh val "âm tính" cho An xem (CPU)
**Mục đích (review của An trên PR #89, chuẩn bị phân tích nhầm lẫn W7):** run 13 báo 38/40 ảnh val âm tính có FP ở conf 0,25, nhưng chỉ có số đếm. Job này lấy **đúng các ảnh JPEG** mà metric run 13 đã dùng (kiểm SHA-256 với `yolo_manifest.csv.gz`), kiểm file nhãn rỗng, rồi xuất:
- `originals/<stem>.jpg`: ảnh gốc 640×512, chép nguyên byte;
- `annotated/<stem>_review.jpg`: vẽ dự đoán run 13 (conf ≥ 0,25: lớp, conf) và hộp GT gần nhất (nét đứt vàng, `GT f<k>`);
- `overview.jpg`: lưới 8×5;
- `review_index.csv`: video, frame, nguyên nhân, nhãn hiện tại (rỗng), dự đoán, hộp GT gần nhất, SHA-256 ảnh.

Không chạy model: dự đoán lấy từ `predictions.json` của run 13 (đã tính sẵn trong repo, `results/W06/val_negative_frames/val_negative_frames.csv`). Không cần GPU.

**Trước khi chạy:**
1. Dataset `e1-w3-tools` → **New Version**, **thêm** từ commit `COMMIT`:
   - `scripts/kaggle/w6_negative_frames_export.py`;
   - `results/W06/val_negative_frames/val_negative_frames.csv`.

   Giữ nguyên các file đang có.
2. *Add Input*: `e1-w3-tools` (bản mới), output notebook `e1-w3-run03b` (chứa `e1-yolo-v1.zip`).
3. *Settings*: Accelerator **None**, Internet **Off** được. Điền `COMMIT`.

**Sau khi chạy:** `w6_negframes.zip` → thư mục kết quả run 16 rồi gửi An. Không đưa ảnh vào Git.


In [ ]:
COMMIT = ""                      # <-- SHA day du 40 ky tu
DATASET = "e1-yolo-v1"
EXPECTED_FP = "d0ac7e230b385132c4bbec90eb5af2c5740a478fec6c79aeb245acca032c27de"   # fingerprint e1-yolo-v1


In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
# Du lieu YOLO: giai nen e1-yolo-v1.zip vao /tmp, kiem SHA zip va fingerprint
import csv, gzip
zips = sorted(glob.glob(f"/kaggle/input/**/{DATASET}.zip", recursive=True))
assert zips, "Khong thay e1-yolo-v1.zip - Add Input: output notebook e1-w3-run03b"
zinfo = json.load(open(find("dataset_zip.json")))
assert sha(zips[0]) == zinfo["sha256"], "SHA-256 cua e1-yolo-v1.zip khac dataset_zip.json"
if not os.path.exists(f"/tmp/e1data/{DATASET}/data.yaml"):
    zipfile.ZipFile(zips[0]).extractall("/tmp/e1data")
DATA_DIR = f"/tmp/e1data/{DATASET}"
card = json.load(open(f"{DATA_DIR}/dataset_card.json"))
assert card["content_fingerprint_sha256"] == EXPECTED_FP, "dataset_card khac fingerprint e1-yolo-v1"
MANIFEST = find("yolo_manifest.csv.gz")
SCRIPT = find("w6_negative_frames_export.py"); INDEX = find("val_negative_frames.csv")
for p in (zips[0], MANIFEST, SCRIPT, INDEX):
    print(f"sha256={sha(p)[:16]}  {p}")
print("DATA_DIR =", DATA_DIR, "| fingerprint", card["content_fingerprint_sha256"][:16], "(phai bat dau d0ac7e23)")


In [ ]:
!{sys.executable} {SCRIPT} --data-dir {DATA_DIR} --manifest {MANIFEST} --index {INDEX} --out /kaggle/working --commit {COMMIT}


## Tự kiểm tra (phải in `PASS`)


In [ ]:
out = "/kaggle/working/w6_negframes"
st = json.load(open(f"{out}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
ex = st.get("export", {}); print("anh:", ex.get("originals"), "| co du doan:", ex.get("with_pred"), "| kich thuoc:", ex.get("sizes"))
ok = all(v["status"] == "OK" for v in st.values()) and len(st) == 4 and ex.get("originals") == 40 and ex.get("with_pred") == 38
print("PASS" if ok else "FAIL - gui stage_status.json cho Claude")
p = "/kaggle/working/w6_negframes.zip"; print("w6_negframes.zip", os.path.getsize(p) // 1024, "KB  sha256", sha(p))
